<a href="https://colab.research.google.com/github/Musarrat1/Assembly/blob/main/tealeaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
print("✅ Memory growth set")

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Memory growth set


In [2]:
import os, glob, numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!ls /content/drive/MyDrive

 classification_report.txt
 Classroom
'Colab Notebooks'
'Cv (2) (1).pdf'
'Cv (2).pdf'
 CV.pdf
 gradcam_outputs
 IMG_3124.JPG
 IMG_3180.JPG
 IMG_3349.JPG
 IMG_3389.JPG
 IMG_3399.JPG
 IMG_3401.JPG
 IMG_3404.JPG
 IMG_3409.JPG
 IMG_3410.JPG
 IMG_3411.JPG
 IMG_3418.JPG
 IMG_3447.JPG
 IMG_3459.JPG
 IMG_3468.JPG
 IMG_3479.JPG
 IMG_3827.JPG
 IMG_3828.JPG
 IMG_3831.JPG
 IMG_3832.JPG
 IMG_3864.JPG
 IMG_3865.JPG
 IMG_4508.JPG
 IMG_4773.JPG
 IMG_4973.JPG
 IMG_5072.JPG
 IMG_5078.JPG
 IMG_5081.JPG
 IMG_5090.JPG
 IMG_5140.JPG
 IMG_5141.JPG
 IMG_5148.JPG
 IMG_5190.JPG
 IMG_5197.JPG
 IMG_5201.JPG
 IMG_5203.JPG
 IMG_5265.JPG
 IMG_5266.JPG
 IMG_5267.JPG
 IMG_5268.JPG
 IMG_5273.JPG
 IMG_5577.JPG
 IMG_5631.JPG
 IMG_5666.JPG
 IMG_5669.JPG
 IMG_5735.JPG
 IMG_5791.JPG
 IMG_5824.JPG
 IMG_5826.JPG
 IMG_5853.JPG
 IMG_5855.JPG
 IMG_5866.JPG
 IMG_6043.JPG
 IMG_6174.JPG
 IMG_6194.JPG
 IMG_6238.JPG
 IMG_6353.JPG
 IMG_6356.JPG
 IMG_6474.JPG
 IMG_6476.JPG
 IMG_6590.JPG
 IMG_6593.JPG
 IMG_7010.JPG
 IMG_7344.JPG
 IMG_73

In [5]:
!cp /content/drive/MyDrive/TeaDataset.zip /content/
!mkdir -p /content/dataset
!unzip -o /content/TeaDataset.zip -d /content/dataset

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5917.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5918.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5919.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust592.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5920.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5921.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5922.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5923.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5924.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5925.jpg  
  inflating: /content/dataset/TeaDataset/TeaDataset/Red rust/Leaf_Red_Rust5926.jpg  
  inflating: /c

In [6]:
import os ,glob
DATA_DIR = "/content/dataset/TeaDataset/TeaDataset"

class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

counts = {}
for c in class_names:
    counts[c] = len(glob.glob(os.path.join(DATA_DIR, c, "*.*")))
print("✅ Class counts:", counts)

✅ Class counts: {'7. Healthy leaf': 13764, 'Brown Blight': 14507, 'Grey Blight': 2024, 'Helopelties': 14289, 'Red rust': 14001}


In [7]:
import numpy as np

total = sum(counts.values())
K = len(class_names)

# balanced weights
class_weights = {i: total/(K*counts[class_names[i]]) for i in range(K)}

# clip to avoid instability (important!)
class_weights_clipped = {k: float(min(v, 1.8)) for k, v in class_weights.items()}

print("Raw weights:", class_weights)
print("✅ Clipped weights:", class_weights_clipped)

Raw weights: {0: 0.8512786980528916, 1: 0.807679051492383, 2: 5.789031620553359, 3: 0.8200013996780741, 4: 0.8368687950860653}
✅ Clipped weights: {0: 0.8512786980528916, 1: 0.807679051492383, 2: 1.8, 3: 0.8200013996780741, 4: 0.8368687950860653}


In [8]:
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = (224, 224)   # safer
BATCH = 8              # FURTHER REDUCED FOR STABILITY
SEED = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="categorical",
    shuffle=False
)

print("✅ TF class order:", train_ds.class_names)  # should match class_names conceptually

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)

Found 58585 files belonging to 5 classes.
Using 46868 files for training.
Found 58585 files belonging to 5 classes.
Using 11717 files for validation.
✅ TF class order: ['7. Healthy leaf', 'Brown Blight', 'Grey Blight', 'Helopelties', 'Red rust']


In [9]:
aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [10]:
import tensorflow as tf
from tensorflow.keras import layers

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = aug(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True), # Re-enabled ModelCheckpoint
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8, # Increased epochs as requested
    class_weight=class_weights_clipped,
    callbacks=callbacks
)


for images, labels in train_ds.take(1):
    plt.figure(figsize=(8,8))
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[tf.argmax(labels[i])])
        plt.axis("off")

Epoch 1/8
5859/5859 ━━━━━━━━━━━━━━━━━━━━ 318s 48ms/step - accuracy: 0.7628 - loss: 0.6558 - val_accuracy: 0.9578 - val_loss: 0.1390 - learning_rate: 1.0000e-05
Epoch 2/8
4090/5859 ━━━━━━━━━━━━━━━━━━━━ 1:13 41ms/step - accuracy: 0.9388 - loss: 0.1705

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True), # Re-enabled ModelCheckpoint
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

USE_CLASS_WEIGHT = True  # set False if accuracy drops or training unstable

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15, # Reverted to 15 epochs for proper training
    class_weight=(class_weights_clipped if USE_CLASS_WEIGHT else None),
    callbacks=callbacks
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from collections import Counter
import numpy as np

# Use the globally available class_names
class_names_for_eval = class_names

y_true, y_pred = [], []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("✅ Pred distribution:", {class_names_for_eval[k]: v for k, v in Counter(y_pred).items()})
print("✅ True distribution:", {class_names_for_eval[k]: v for k, v in Counter(y_true).items()})
print("\n✅ Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names_for_eval, digits=4))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names_for_eval)

plt.figure(figsize=(8,8))
disp.plot(values_format='d', xticks_rotation=45)
plt.title("Confusion Matrix (Validation)")
plt.show()

In [ ]:
remedy = {
    "7. Healthy leaf": "Healthy: maintain field hygiene, regular monitoring, balanced nutrition.",
    "Brown Blight": "Brown blight: remove infected leaves, improve airflow, avoid overhead watering; fungicide if severe.",
    "Grey Blight": "Grey blight: prune infected parts, reduce humidity, maintain spacing; fungicide if severe.",
    "Red Rust": "Red rust (algal leaf spot): reduce shading, improve sunlight; copper-based spray if needed.",
    "Helopelties": "Helopeltis (tea mosquito bug): IPM—monitoring, pruning, neem-based spray/approved control when infestation increases."
}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

images, labels = next(iter(val_ds))
img = images[0]

pred = model.predict(np.expand_dims(img.numpy(), axis=0), verbose=0)
pred_label = int(np.argmax(pred[0]))
pred_class = class_names[pred_label] # Corrected from class_names_tf

plt.figure(figsize=(4,4))
plt.imshow(img.numpy().astype("uint8"))
plt.axis("off")
plt.title(f"Pred: {pred_class}")
plt.show()

print("Predicted:", pred_class)
print("Remedy:", remedy.get(pred_class, "Consult an agriculture expert."))

In [ ]:
model.save("tea_leaf_model_final.keras")
!cp tea_leaf_model_final.keras /content/drive/MyDrive/
print("✅ Saved: /content/drive/MyDrive/tea_leaf_model_final.keras")